# Experiment 2 — LocalOracle-PC

Full structured implementation of Experiment 2 from `proposal/COTCE_extended_exp.md`. The model receives a source, retains only a hard `d × p` state, and can read at most one function row per update. Requested depth `D` is a legal shared continuation. A hard `STOP` action leaves the state unchanged and consumes no oracle row.

This notebook is a structured theorem-facing implementation. It is not the unrestricted ProsQA model and does not expose the full function table to the controller.

In [ ]:
from __future__ import annotations

import gc, hashlib, json, math, random, time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from finite_cot.access import LocalOracleAccess
from finite_cot.models import BinaryPointerMachine
from finite_cot.quantization import FiniteScalarQuantizer, QuantizedTensor

ROOT = Path.cwd().resolve()
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = ROOT / 'results' / 'local_oracle_pc_exp2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('root:', ROOT, '| device:', DEVICE)

In [ ]:
# Cell 2 — fixed checkpoint registry. Every checkpoint owns its exact d and p.
# TRAIN_MISSING=True creates structured pilot checkpoints at these paths.
MODEL_SPECS = [
    {'name':'b4-d4-p1',   'model_path':'ckpts/local_oracle_exp2/b4-d4-p1.pt',   'd':4,  'p':1, 'seed':17},
    {'name':'b4-d2-p2',   'model_path':'ckpts/local_oracle_exp2/b4-d2-p2.pt',   'd':2,  'p':2, 'seed':17},
    {'name':'b4-d1-p4',   'model_path':'ckpts/local_oracle_exp2/b4-d1-p4.pt',   'd':1,  'p':4, 'seed':17},
    {'name':'b8-d8-p1',   'model_path':'ckpts/local_oracle_exp2/b8-d8-p1.pt',   'd':8,  'p':1, 'seed':17},
    {'name':'b8-d4-p2',   'model_path':'ckpts/local_oracle_exp2/b8-d4-p2.pt',   'd':4,  'p':2, 'seed':17},
    {'name':'b8-d2-p4',   'model_path':'ckpts/local_oracle_exp2/b8-d2-p4.pt',   'd':2,  'p':4, 'seed':17},
    {'name':'b8-d1-p8',   'model_path':'ckpts/local_oracle_exp2/b8-d1-p8.pt',   'd':1,  'p':8, 'seed':17},
    {'name':'b16-d16-p1', 'model_path':'ckpts/local_oracle_exp2/b16-d16-p1.pt', 'd':16, 'p':1, 'seed':17},
    {'name':'b16-d8-p2',  'model_path':'ckpts/local_oracle_exp2/b16-d8-p2.pt',  'd':8,  'p':2, 'seed':17},
    {'name':'b16-d4-p4',  'model_path':'ckpts/local_oracle_exp2/b16-d4-p4.pt',  'd':4,  'p':4, 'seed':17},
    {'name':'b16-d2-p8',  'model_path':'ckpts/local_oracle_exp2/b16-d2-p8.pt',  'd':2,  'p':8, 'seed':17},
]

MAX_NODES = 512
TRAIN_NODE_LIMIT = 128
HIDDEN_DIM = 64
TRAIN_MISSING = True
TEST_MODE = True

T_VALUES_FULL = [0, 1, 2, 4, 6, 8, 12, 16]
N_VALUES_FULL = [64, 128, 256, 512]
D_VALUES_FULL = [1, 2, 4, 6, 8, 12, 16]
SEEDS_FULL = [17, 42, 137, 314, 2718]

if TEST_MODE:
    RUN_SPECS = [MODEL_SPECS[0], MODEL_SPECS[3]]
    T_VALUES, N_VALUES, D_VALUES, EVAL_PER_CELL = [0,1,2,4], [64], [2,4], 256
    TRAIN_EPOCHS = 20
    MECHANISM_PAIRS = 128
else:
    RUN_SPECS = MODEL_SPECS
    T_VALUES, N_VALUES, D_VALUES, EVAL_PER_CELL = T_VALUES_FULL, N_VALUES_FULL, D_VALUES_FULL, 10_000
    TRAIN_EPOCHS = 100
    MECHANISM_PAIRS = 5_000

pd.DataFrame(MODEL_SPECS).assign(state_bits=lambda x: x.d*x.p)

## Hard finite-state controller

`DepthAwarePointerMachine` never receives the function table. Its hard action is a row index or `STOP`. Only the selected response is encoded into the next hard state. The legal continuation contributes only the Boolean fact `steps_remaining > 0`; it is not persistent memory.

In [ ]:
@dataclass
class Rollout:
    prediction: torch.Tensor
    actions: list[torch.Tensor]
    responses: list[torch.Tensor]
    state_values: list[torch.Tensor]
    state_codes: list[torch.Tensor]
    row_reads: int
    stop_steps: list[torch.Tensor]

class DepthAwarePointerMachine(nn.Module):
    def __init__(self, max_nodes, state_dim, bits, hidden_dim=64):
        super().__init__()
        self.max_nodes, self.state_dim, self.bits = int(max_nodes), int(state_dim), int(bits)
        self.stop_id = self.max_nodes
        self.node_state = nn.Embedding(self.max_nodes, self.state_dim)
        self.quantizer = FiniteScalarQuantizer(self.bits, 1.0)
        self.continue_embedding = nn.Embedding(2, hidden_dim)
        self.action_head = nn.Sequential(nn.Linear(self.state_dim+hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, self.max_nodes+1))
        self.vertex_head = nn.Sequential(nn.Linear(self.state_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, self.max_nodes))

    def encode(self, vertices):
        return self.quantizer.quantize(self.node_state(vertices))

    def vertex_logits(self, state, n):
        logits = self.vertex_head(state)
        return logits[:, :n]

    def action_logits(self, state, should_continue, n):
        flag = self.continue_embedding(should_continue.long())
        logits = self.action_head(torch.cat((state, flag), dim=-1))
        return torch.cat((logits[:, :n], logits[:, self.stop_id:self.stop_id+1]), dim=-1)

    @torch.inference_mode()
    def rollout(self, function_table, source, depth, max_updates, initial_state=None):
        batch, n = function_table.shape
        q = self.encode(source) if initial_state is None else initial_state
        state, codes = q.value, [q.codes]
        states, actions, responses, stop_steps = [state], [], [], []
        row_reads = 0
        for step in range(max_updates):
            remaining = torch.full((batch,), int(step < depth), device=state.device, dtype=torch.long)
            action_local = self.action_logits(state, remaining, n).argmax(-1)
            is_stop = action_local.eq(n)
            stop_steps.append(is_stop)
            response = torch.full_like(source, -1)
            active = (~is_stop).nonzero(as_tuple=False).flatten()
            if active.numel():
                # Build the oracle only for active examples. STOP examples perform
                # no table access at all, including no discarded dummy lookup.
                active_oracle = LocalOracleAccess(function_table[active])
                active_query = action_local[active]
                active_oracle.begin_update()
                observed = active_oracle.query(active_query)
                active_oracle.end_update()
                response[active] = observed
                updated = self.encode(observed)
                state = state.clone(); state[active] = updated.value
                code = codes[-1].clone(); code[active] = updated.codes
                row_reads += int(active.numel())
            else:
                code = codes[-1]
            actions.append(torch.where(is_stop, torch.full_like(action_local, self.stop_id), action_local))
            responses.append(torch.where(is_stop, torch.full_like(response, -1), response))
            codes.append(code); states.append(state)
        prediction = self.vertex_logits(state, n).argmax(-1)
        return Rollout(prediction, actions, responses, states, codes, row_reads, stop_steps)


## Train or load each fixed `(d,p)` checkpoint

The included trainer is an explicitly labeled **intermediate-vertex/oracle-supervision upper bound**. It teaches the hard state to decode the current vertex and teaches the hard action to query that vertex until `STOP`. Replace registry paths with answer-only trained checkpoints for the proposal's primary learned tier.

In [ ]:
def train_structured_checkpoint(spec):
    torch.manual_seed(int(spec['seed']))
    model = DepthAwarePointerMachine(MAX_NODES, spec['d'], spec['p'], HIDDEN_DIM).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
    ids = torch.arange(TRAIN_NODE_LIMIT)
    flags = torch.tensor([0,1])
    nodes = ids.repeat_interleave(2)
    continuing = flags.repeat(TRAIN_NODE_LIMIT)
    loader = DataLoader(TensorDataset(nodes, continuing), batch_size=128, shuffle=True)
    for _ in tqdm(range(TRAIN_EPOCHS), desc=f"train {spec['name']}", leave=False):
        model.train()
        for vertex, flag in loader:
            vertex, flag = vertex.to(DEVICE), flag.to(DEVICE)
            state = model.encode(vertex).value
            vertex_loss = F.cross_entropy(model.vertex_logits(state, MAX_NODES), vertex)
            action_target = torch.where(flag.bool(), vertex, torch.full_like(vertex, MAX_NODES))
            action_loss = F.cross_entropy(model.action_logits(state, flag, MAX_NODES), action_target)
            loss = vertex_loss + action_loss
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
    path=ROOT/spec['model_path']; path.parent.mkdir(parents=True,exist_ok=True)
    torch.save({'state_dict':model.state_dict(),'spec':spec,'supervision':'intermediate_vertex_oracle'},path)
    return model.eval()

def load_model(spec):
    path=ROOT/spec['model_path']
    if not path.exists():
        if not TRAIN_MISSING: raise FileNotFoundError(path)
        return train_structured_checkpoint(spec)
    model=DepthAwarePointerMachine(MAX_NODES,spec['d'],spec['p'],HIDDEN_DIM)
    blob=torch.load(path,map_location='cpu',weights_only=False)
    state=blob.get('state_dict',blob) if isinstance(blob,dict) else blob
    model.load_state_dict(state,strict=True)
    return model.to(DEVICE).eval()


## Deterministic data manifest

Each example contains a planted simple path of length `D`; all nonpath rows are filled independently. Seeds are functions of `(evaluation seed,n,D)`, so cells are reproducible and do not overlap accidentally.

In [ ]:
def make_pointer_batch(n, depth, count, seed, device=DEVICE):
    if depth >= n: raise ValueError('depth must be smaller than n')
    g=torch.Generator().manual_seed(int(seed))
    function=torch.randint(0,n,(count,n),generator=g)
    source=torch.empty(count,dtype=torch.long)
    paths=torch.empty(count,depth+1,dtype=torch.long)
    for row in range(count):
        path=torch.randperm(n,generator=g)[:depth+1]
        function[row,path[:-1]]=path[1:]
        source[row]=path[0]; paths[row]=path
    return function.to(device),source.to(device),paths.to(device)

def cell_seed(seed,n,depth): return int(seed*1_000_003+n*10_007+depth*101)


## Primary grid: accuracy over both threshold margins

The result table records `T-D` and `dp-ceil(log2 n)`, row reads, STOP behavior, and decoded-current-vertex accuracy. No evaluation result is cached or reused.

In [ ]:
@torch.inference_mode()
def evaluate_cell(model,spec,n,depth,T,seed,count):
    function,source,path=make_pointer_batch(n,depth,count,cell_seed(seed,n,depth))
    out=model.rollout(function,source,depth,T)
    target=path[:,-1]
    step_scores=[]
    for step,state in enumerate(out.state_values):
        expected=path[:,min(step,depth)]
        step_scores.append(float(model.vertex_logits(state,n).argmax(-1).eq(expected).float().mean()))
    stopped=torch.stack(out.stop_steps,1) if out.stop_steps else torch.empty(count,0,dtype=torch.bool,device=DEVICE)
    premature=float(stopped[:,:min(T,depth)].float().mean()) if min(T,depth)>0 else 0.0
    post_stop=float(stopped[:,depth:].float().mean()) if T>depth else float('nan')
    return {
        'model':spec['name'],'seed':seed,'n':n,'D':depth,'T':T,'d':spec['d'],'p':spec['p'],
        'state_bits':spec['d']*spec['p'],'required_bits':math.ceil(math.log2(n)),
        'T_minus_D':T-depth,'bits_minus_required':spec['d']*spec['p']-math.ceil(math.log2(n)),
        'accuracy':float(out.prediction.eq(target).float().mean()),'examples':count,
        'row_reads_per_example':out.row_reads/count,'premature_stop_rate':premature,
        'post_depth_stop_rate':post_stop,'decoded_vertex_accuracy_by_step':step_scores,
    }

primary_rows=[]
for spec in RUN_SPECS:
    model=load_model(spec)
    for seed in ([17] if TEST_MODE else SEEDS_FULL):
        for n in N_VALUES:
            for depth in D_VALUES:
                for T in T_VALUES:
                    primary_rows.append(evaluate_cell(model,spec,n,depth,T,seed,EVAL_PER_CELL))
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
primary=pd.DataFrame(primary_rows)
primary.to_csv(OUTPUT_DIR/'primary_surface.csv',index=False)
display(primary.head())

## Primary plots and transition checks

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

summary=primary.groupby(['T_minus_D','bits_minus_required'],as_index=False).accuracy.mean()
fig,ax=plt.subplots(figsize=(8,6))
pivot=summary.pivot(index='bits_minus_required',columns='T_minus_D',values='accuracy')
sns.heatmap(pivot,annot=True,fmt='.2f',vmin=0,vmax=1,cmap='viridis',ax=ax)
ax.set_title('LocalOracle-PC accuracy surface'); fig.tight_layout(); plt.show()

fig,ax=plt.subplots(figsize=(9,6))
sns.lineplot(data=primary,x='T_minus_D',y='accuracy',hue='bits_minus_required',marker='o',errorbar=('ci',95),ax=ax)
ax.axvline(0,color='black',ls='--',alpha=.5); ax.set_ylim(0,1.02); ax.grid(alpha=.25); fig.tight_layout(); plt.show()

sufficient=primary[primary.bits_minus_required>=0]
checks={
 'no_capacity_compensation_below_depth': float(primary[primary.T_minus_D<0].accuracy.max()),
 'mean_gain_Dminus1_to_D': float(
     sufficient[sufficient.T_minus_D.eq(0)].accuracy.mean()-sufficient[sufficient.T_minus_D.eq(-1)].accuracy.mean()
 ) if {-1,0}.issubset(set(sufficient.T_minus_D)) else np.nan,
 'joint_ood_accuracy': float(primary[(primary.n==512)&(primary.D==16)&(primary.T==16)].accuracy.mean()) if not TEST_MODE else np.nan,
}
display(checks)

## Current-vertex decoding and state interchange

For interchange, donor and recipient use the same function but different sources. At step `t`, the recipient state is replaced by the donor state, then execution continues for `D-t` hops. The target is computed directly as `f^(D-t)(v_t_donor)`.

In [ ]:
def apply_function(function,vertices,steps):
    rows=torch.arange(function.shape[0],device=function.device)
    value=vertices
    for _ in range(steps): value=function[rows,value]
    return value

@torch.inference_mode()
def state_interchange(model,spec,n=64,depth=4,t=2,pairs=1000,seed=991):
    g=torch.Generator().manual_seed(seed)
    function=torch.randint(0,n,(pairs,n),generator=g).to(DEVICE)
    recipient=torch.randint(0,n,(pairs,),generator=g).to(DEVICE)
    donor=torch.randint(0,n,(pairs,),generator=g).to(DEVICE)
    donor_prefix=model.rollout(function,donor,depth,t)
    donor_vertex=apply_function(function,donor,t)
    target=apply_function(function,donor_vertex,depth-t)
    patched=model.rollout(function,recipient,depth-t,depth-t,initial_state=QuantizedTensor(donor_prefix.state_values[-1], donor_prefix.state_codes[-1]))
    return {'model':spec['name'],'n':n,'D':depth,'patch_step':t,'pairs':pairs,
            'state_interchange_accuracy':float(patched.prediction.eq(target).float().mean())}

interchange=[]
for spec in RUN_SPECS:
    model=load_model(spec)
    for t in ([1,2,3] if not TEST_MODE else [2]):
        interchange.append(state_interchange(model,spec,pairs=MECHANISM_PAIRS,t=t))
interchange=pd.DataFrame(interchange)
interchange.to_csv(OUTPUT_DIR/'state_interchange.csv',index=False)
display(interchange)

## Causal light cone, no-op, and forced stopping

In [ ]:
@torch.inference_mode()
def mechanism_checks(model,spec,n=64,depth=4,count=512,seed=717):
    function,source,path=make_pointer_batch(n,depth,count,seed)
    base=model.rollout(function,source,depth,depth+1)
    # Change the response at hop j; states through j must remain identical.
    j=min(2,depth-1); changed=function.clone(); rows=torch.arange(count,device=DEVICE)
    changed[rows,path[:,j]]=(path[:,j+1]+1)%n
    altered=model.rollout(changed,source,depth,depth+1)
    hashes_equal_through_j=all(torch.equal(base.state_codes[k],altered.state_codes[k]) for k in range(j+1))
    diverged_after=float(base.state_codes[j+1].ne(altered.state_codes[j+1]).any(1).float().mean())
    # A legal no-response update leaves the hard state unchanged by definition.
    state=base.state_values[min(j,len(base.state_values)-1)]
    before=model.vertex_logits(state,n).argmax(-1); after=model.vertex_logits(state.clone(),n).argmax(-1)
    no_op_advance=float(before.ne(after).float().mean())
    forced={}
    for stop_at in (depth-1,depth,depth+1):
        out=model.rollout(function,source,depth,max(0,stop_at))
        forced[str(stop_at)]=float(out.prediction.eq(path[:,-1]).float().mean())
    generator=torch.Generator(device=DEVICE).manual_seed(seed+1)
    permutation=torch.randperm(n,generator=generator,device=DEVICE); inverse=torch.argsort(permutation)
    relabeled_function=permutation[function[:,inverse]]
    relabeled=model.rollout(relabeled_function,permutation[source],depth,depth)
    relabeling_accuracy=float(relabeled.prediction.eq(permutation[path[:,-1]]).float().mean())
    return {'model':spec['name'],'light_cone_equal_through_j':hashes_equal_through_j,
            'light_cone_divergence_after_j':diverged_after,'no_op_advance_rate':no_op_advance,
            'heldout_relabeling_accuracy':relabeling_accuracy,'forced_stop_accuracy':forced}

mechanisms=[]
for spec in RUN_SPECS:
    model=load_model(spec); mechanisms.append(mechanism_checks(model,spec,count=MECHANISM_PAIRS))
display(pd.DataFrame(mechanisms))

## Paired-completion impossibility audit for `T < D`

The online oracle answers both worlds identically. Whenever the model queries the current unassigned path endpoint, it returns a fresh vertex; unrelated new queries become self-loops. After `T` queries, the still-unqueried endpoint is completed differently in worlds A and B. The complete observed transcript and hard computation are identical, while the two `D`-step targets differ.

In [ ]:
@torch.inference_mode()
def paired_completion_trial(model,n,D,T,seed):
    if not T<D: raise ValueError('paired completion requires T < D')
    rng=random.Random(seed); source=0; assigned={}; used={source}; endpoint=source
    q=model.encode(torch.tensor([source],device=DEVICE)); state=q.value
    transcript=[]
    for step in range(T):
        cont=torch.ones(1,dtype=torch.long,device=DEVICE)
        local=int(model.action_logits(state,cont,n).argmax())
        query=0 if local==n else local
        if query not in assigned:
            if query==endpoint:
                fresh=next(v for v in range(n) if v not in used and v not in assigned)
                assigned[query]=fresh; used.add(fresh); endpoint=fresh
            else: assigned[query]=query
        response=assigned[query]; transcript.append((query,response))
        state=model.encode(torch.tensor([response],device=DEVICE)).value
    y=next(v for v in range(n) if v not in used and v not in assigned and v!=endpoint)
    fa=torch.arange(n); fb=torch.arange(n)
    for key,value in assigned.items(): fa[key]=value; fb[key]=value
    fa[endpoint]=endpoint; fb[endpoint]=y; fb[y]=y
    def endpoint_of(f):
        v=source
        for _ in range(D): v=int(f[v])
        return v
    target_a,target_b=endpoint_of(fa),endpoint_of(fb)
    prediction=int(model.vertex_logits(state,n).argmax())
    return transcript,target_a,target_b,prediction

def paired_completion_audit(model,spec,n=512,D=16,T=8,trials=500):
    failures=0; valid=0
    for i in range(trials):
        transcript,a,b,pred=paired_completion_trial(model,n,D,T,10_000+i)
        valid+=int(a!=b); failures+=int(pred!=a or pred!=b)
    return {'model':spec['name'],'n':n,'D':D,'T':T,'trials':trials,
            'different_endpoint_rate':valid/trials,'at_least_one_world_wrong_rate':failures/trials}

paired=[]
for spec in RUN_SPECS:
    model=load_model(spec)
    for D in ([4] if TEST_MODE else [4,8,12,16]):
        for T in [t for t in T_VALUES_FULL if t<D]:
            paired.append(paired_completion_audit(model,spec,D=D,T=T,trials=MECHANISM_PAIRS))
paired=pd.DataFrame(paired); paired.to_csv(OUTPUT_DIR/'paired_completion.csv',index=False); display(paired)

## Controls and held-out node relabeling

The pause control makes the same number of controller decisions but reads no rows. The full-table solver is explicitly outside the local-oracle model. Teacher-forced access is an oracle-supervision upper bound. The natural binary trace reports `L_required = D ceil(log2 n)` as accounting, not learned discrete-CoT accuracy.


In [ ]:
@torch.inference_mode()
def evaluate_controls(model,spec,n,D,T,count=1000,seed=818):
    function,source,path=make_pointer_batch(n,D,count,seed)
    target=path[:,-1]
    initial=model.encode(source).value
    pause_prediction=model.vertex_logits(initial,n).argmax(-1)
    full_table_prediction=apply_function(function,source,D)
    teacher_state=model.encode(target).value
    teacher_prediction=model.vertex_logits(teacher_state,n).argmax(-1)

    # One held-out global relabeling pi: f_pi(pi(v)) = pi(f(v)).
    generator=torch.Generator(device=DEVICE).manual_seed(seed+1)
    permutation=torch.randperm(n,generator=generator,device=DEVICE)
    inverse=torch.argsort(permutation)
    relabeled_function=permutation[function[:,inverse]]
    relabeled_source=permutation[source]
    relabeled_target=permutation[target]
    relabeled=model.rollout(relabeled_function,relabeled_source,D,T)

    return {
        'model':spec['name'],'n':n,'D':D,'T':T,'d':spec['d'],'p':spec['p'],
        'pause_no_oracle_accuracy':float(pause_prediction.eq(target).float().mean()),
        'pause_row_reads':0,
        'full_table_accuracy_outside_model':float(full_table_prediction.eq(target).float().mean()),
        'teacher_forced_state_accuracy':float(teacher_prediction.eq(target).float().mean()),
        'heldout_relabeling_accuracy':float(relabeled.prediction.eq(relabeled_target).float().mean()),
        'binary_trace_L_required':int(D*math.ceil(math.log2(n))),
    }

control_rows=[]
for spec in RUN_SPECS:
    model=load_model(spec)
    for n in N_VALUES:
        for D in D_VALUES:
            control_rows.append(evaluate_controls(model,spec,n,D,max(T_VALUES),256 if TEST_MODE else 1000))
controls=pd.DataFrame(control_rows)
controls.to_csv(OUTPUT_DIR/'controls.csv',index=False)
display(controls)


## Fixed binary construction ceiling and final decision table

The fixed construction is reported separately from learned evidence. It should be exact exactly when `d >= ceil(log2 n)` and the allowed execution reaches `D`.

In [ ]:
@torch.inference_mode()
def fixed_binary_grid(count=1000):
    rows=[]
    for n in N_VALUES:
        for D in D_VALUES:
            function,source,path=make_pointer_batch(n,D,count,88+n+D)
            for d in sorted({max(1,math.ceil(math.log2(n))-1),math.ceil(math.log2(n))}):
                machine=BinaryPointerMachine(d)
                for T in T_VALUES:
                    executed=min(T,D)
                    try: prediction=machine.run(function,source,executed).vertex
                    except IndexError: prediction=torch.full_like(source,-1)
                    rows.append({'n':n,'D':D,'T':T,'d':d,'accuracy':float(prediction.eq(path[:,-1]).float().mean()),
                                 'capacity_sufficient':d>=math.ceil(math.log2(n)),'depth_sufficient':T>=D})
    return pd.DataFrame(rows)
fixed=fixed_binary_grid(256 if TEST_MODE else 1000)
fixed.to_csv(OUTPUT_DIR/'fixed_binary_ceiling.csv',index=False)

decision={
 'capacity_cannot_compensate_for_T_below_D': bool(primary[primary.T_minus_D<0].accuracy.max()<0.99),
 'transition_near_T_equals_D_at_sufficient_capacity': checks['mean_gain_Dminus1_to_D'],
 'state_interchange_mean_accuracy': float(interchange.state_interchange_accuracy.mean()),
 'paired_completion_pass': bool(paired.at_least_one_world_wrong_rate.eq(1.0).all()),
 'no_op_pass': bool(all(x['no_op_advance_rate']==0 for x in mechanisms)),
 'interpretation': 'A convincing mechanism requires the first three scientific conditions; fixed binary results are construction checks only.',
}
display(decision)
(OUTPUT_DIR/'decision.json').write_text(json.dumps(decision,indent=2))